# Diagnostico de modelo de regresion

Este notebook analiza un modelo YA entrenado para responder tres preguntas:
1. Que tan bien predice (metricas globales).
2. Donde falla (segmentos y casos de mayor error).
3. Por que falla (importancia de variables y SHAP opcional).

## 0) Checklist rapido antes de correr

- Tener un modelo guardado en formato `.joblib` o `.pkl`.
- Tener un dataset de evaluacion con la variable target y las features.
- Confirmar nombre de la target y columnas de segmentacion disponibles.
- Si quieres explicabilidad avanzada, instala SHAP en tu entorno.

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [3]:
# =============================
# Configuracion del diagnostico
# =============================

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'Doback-Data').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd().resolve())

# Rutas configuradas para el modelo encontrado
MODEL_PATH = REPO_ROOT / 'output' / 'models' / 'w_model_models.joblib'
EVAL_DATA_PATH = REPO_ROOT / 'Doback-Data' / 'featured' / 'DOBACK024_20250929_seg10.csv'
TARGET_COL = 'gz'

# Si tu modelo NO guarda feature_names_in_, define una lista explicita aqui
FEATURES_OVERRIDE = None

# Columnas para analizar donde falla el modelo
SEGMENT_COL_CANDIDATES = ['fecha', 'road_name', 'highway']
SPEED_COL = 'speed_kmh'

TOP_K_WORST = 25
MIN_GROUP_SIZE = 80
N_PERMUTATION_REPEATS = 8
N_SHAP_SAMPLES = 2000

print('MODEL_PATH   :', MODEL_PATH)
print('EVAL_DATA_PATH:', EVAL_DATA_PATH)
print('TARGET_COL   :', TARGET_COL)

MODEL_PATH   : C:\Users\alexc\Nextcloud\phd-AlexCastilla\code\LiDAR-Stability-algorithm\output\models\w_model_models.joblib
EVAL_DATA_PATH: C:\Users\alexc\Nextcloud\phd-AlexCastilla\code\LiDAR-Stability-algorithm\Doback-Data\featured\DOBACK024_20250929_seg10.csv
TARGET_COL   : gz


In [5]:
def read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'No existe el archivo: {path}')

    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path, low_memory=False)
    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(path)

    raise ValueError(f'Formato no soportado: {suffix}. Usa .csv o .parquet')


def load_model(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'No existe el modelo: {path}')

    suffix = path.suffix.lower()
    if suffix not in {'.joblib', '.pkl'}:
        raise ValueError('El modelo debe ser .joblib o .pkl')

    return joblib.load(path)


def infer_feature_columns(model, df: pd.DataFrame, target_col: str, override=None):
    if override is not None:
        return list(override)

    if hasattr(model, 'feature_names_in_'):
        return list(model.feature_names_in_)

    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target_col]
    if not numeric_cols:
        raise ValueError('No se pudieron inferir features numericas')
    return numeric_cols


model = load_model(Path(MODEL_PATH))
df_eval = read_table(Path(EVAL_DATA_PATH))

if TARGET_COL not in df_eval.columns:
    raise ValueError(f'La target {TARGET_COL} no existe en el dataset de evaluacion')

feature_cols = infer_feature_columns(model, df_eval, TARGET_COL, FEATURES_OVERRIDE)
missing_features = [c for c in feature_cols if c not in df_eval.columns]
if missing_features:
    raise ValueError(f'Faltan features en evaluacion: {missing_features[:20]}')

work_cols = feature_cols + [TARGET_COL]
df_work = df_eval[work_cols].dropna().copy()

X_eval = df_work[feature_cols]
y_true = df_work[TARGET_COL]
y_pred = pd.Series(model.predict(X_eval), index=df_work.index, name='y_pred')

print('Filas dataset original :', len(df_eval))
print('Filas usadas en analisis:', len(df_work))
print('Numero de features      :', len(feature_cols))

ValueError: EOF: reading array data, expected 262144 bytes got 66336

In [ ]:
def safe_mape(y_t: pd.Series, y_p: pd.Series, eps: float = 1e-6) -> float:
    denom = np.maximum(np.abs(y_t), eps)
    return float(np.mean(np.abs(y_t - y_p) / denom) * 100.0)


def regression_metrics(y_t: pd.Series, y_p: pd.Series) -> pd.DataFrame:
    residuals = y_p - y_t
    abs_err = np.abs(residuals)

    metrics = {
        'rmse': float(np.sqrt(mean_squared_error(y_t, y_p))),
        'mae': float(mean_absolute_error(y_t, y_p)),
        'medae': float(median_absolute_error(y_t, y_p)),
        'r2': float(r2_score(y_t, y_p)),
        'mape_percent': safe_mape(y_t, y_p),
        'bias_mean_error': float(np.mean(residuals)),
        'p90_abs_error': float(np.percentile(abs_err, 90)),
        'p95_abs_error': float(np.percentile(abs_err, 95)),
        'max_abs_error': float(np.max(abs_err)),
        'n_samples': int(len(y_t)),
    }
    return pd.DataFrame({'metric': list(metrics.keys()), 'value': list(metrics.values())})


metrics_df = regression_metrics(y_true, y_pred)
display(metrics_df)

In [ ]:
# Graficos base de diagnostico
residuals = y_pred - y_true
abs_error = residuals.abs()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(y_true, y_pred, s=8, alpha=0.3)
mn = min(y_true.min(), y_pred.min())
mx = max(y_true.max(), y_pred.max())
axes[0, 0].plot([mn, mx], [mn, mx], color='red', linestyle='--')
axes[0, 0].set_title('y_true vs y_pred')
axes[0, 0].set_xlabel('y_true')
axes[0, 0].set_ylabel('y_pred')

sns.histplot(residuals, bins=60, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribucion de residuales')

axes[1, 0].scatter(y_pred, residuals, s=8, alpha=0.3)
axes[1, 0].axhline(0, color='red', linestyle='--')
axes[1, 0].set_title('Residuales vs prediccion')
axes[1, 0].set_xlabel('y_pred')
axes[1, 0].set_ylabel('residual')

sns.histplot(abs_error, bins=60, kde=True, ax=axes[1, 1], color='orange')
axes[1, 1].set_title('Distribucion de error absoluto')

plt.tight_layout()
plt.show()

In [ ]:
# Analisis de fallos por segmentos
df_diag = df_eval.loc[df_work.index].copy()
df_diag['y_true'] = y_true.values
df_diag['y_pred'] = y_pred.values
df_diag['residual'] = df_diag['y_pred'] - df_diag['y_true']
df_diag['abs_error'] = df_diag['residual'].abs()

if SPEED_COL in df_diag.columns:
    df_diag['speed_bin'] = pd.cut(
        df_diag[SPEED_COL],
        bins=[-0.1, 20, 40, 60, 90, np.inf],
        labels=['0-20', '20-40', '40-60', '60-90', '90+']
    )


def group_error_report(df: pd.DataFrame, group_col: str, min_n: int = 50) -> pd.DataFrame:
    if group_col not in df.columns:
        return pd.DataFrame()

    rep = (
        df.groupby(group_col, dropna=False)
        .agg(
            n=('abs_error', 'size'),
            mae=('abs_error', 'mean'),
            rmse=('residual', lambda s: float(np.sqrt(np.mean(np.square(s))))),
            bias=('residual', 'mean'),
            p95_abs=('abs_error', lambda s: float(np.percentile(s, 95))),
        )
        .reset_index()
    )

    rep = rep[rep['n'] >= min_n].sort_values(['mae', 'rmse'], ascending=False)
    return rep


segment_reports = {}
for col in SEGMENT_COL_CANDIDATES + (['speed_bin'] if 'speed_bin' in df_diag.columns else []):
    report = group_error_report(df_diag, col, min_n=MIN_GROUP_SIZE)
    if not report.empty:
        segment_reports[col] = report
        print(f'
Top segmentos problematicos para: {col}')
        display(report.head(12))

if not segment_reports:
    print('No se pudieron construir reportes por segmentos con las columnas candidatas.')

In [ ]:
# Casos con mayor error absoluto (inspeccion directa)
worst_cases = df_diag.sort_values('abs_error', ascending=False).head(TOP_K_WORST)
cols_for_review = [c for c in [TARGET_COL, 'y_pred', 'abs_error', SPEED_COL, 'phi_lidar', 'tri', 'ruggedness', 'road_name', 'fecha'] if c in worst_cases.columns]
display(worst_cases[cols_for_review])

In [ ]:
# Importancia global: MDI del modelo y permutation importance
mdi_df = pd.DataFrame()
if hasattr(model, 'feature_importances_'):
    mdi_df = pd.DataFrame({
        'feature': feature_cols,
        'importance_mdi': model.feature_importances_,
    }).sort_values('importance_mdi', ascending=False)
    print('Top importancia MDI')
    display(mdi_df.head(20))

perm = permutation_importance(
    model,
    X_eval,
    y_true,
    n_repeats=N_PERMUTATION_REPEATS,
    random_state=42,
    n_jobs=-1,
)
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_perm_mean': perm.importances_mean,
    'importance_perm_std': perm.importances_std,
}).sort_values('importance_perm_mean', ascending=False)

print('Top permutation importance')
display(perm_df.head(20))

plt.figure(figsize=(10, 8))
sns.barplot(data=perm_df.head(15), x='importance_perm_mean', y='feature', palette='viridis')
plt.title('Top 15 - Permutation importance')
plt.xlabel('Mean importance drop')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP opcional (global y local)
shap_available = True
try:
    import shap
except Exception as ex:
    shap_available = False
    print('SHAP no disponible. Instala con: pip install shap')
    print('Detalle:', ex)

if shap_available:
    sample_n = min(N_SHAP_SAMPLES, len(X_eval))
    X_shap = X_eval.sample(n=sample_n, random_state=42) if len(X_eval) > sample_n else X_eval.copy()

    # Para modelos de arboles, TreeExplainer suele ser la opcion mas estable
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_shap)
    except Exception:
        explainer = shap.Explainer(model.predict, X_shap)
        shap_values = explainer(X_shap)

    print('Resumen SHAP global')
    shap.summary_plot(shap_values, X_shap, show=True)

    top_features = perm_df.head(3)['feature'].tolist() if 'perm_df' in globals() else feature_cols[:3]
    for feat in top_features:
        if feat in X_shap.columns:
            shap.dependence_plot(feat, shap_values, X_shap, show=True)

    # Explicacion local sobre casos de mayor error
    print('Explicacion local SHAP en casos con mayor error')
    worst_idx = df_diag.sort_values('abs_error', ascending=False).head(3).index
    X_local = X_eval.loc[worst_idx]

    try:
        local_vals = explainer.shap_values(X_local) if hasattr(explainer, 'shap_values') else explainer(X_local)
        shap.summary_plot(local_vals, X_local, show=True)
    except Exception as ex:
        print('No se pudo generar SHAP local:', ex)

## Plantilla de conclusiones accionables

Usa esta estructura al cerrar el analisis:

1. **Que tan bien funciona**: resume RMSE, MAE, R2, sesgo, p95 error.
2. **Donde falla**: lista top segmentos por MAE/RMSE y top casos de error.
3. **Por que falla**: cruza SHAP/permutation con segmentos para identificar causas probables.
4. **Que cambiar primero**: prioriza acciones en datos, features y modelo.
5. **Como validar la mejora**: define test A/B o comparacion antes/despues con mismas metricas.